# Exploración de datos de Instacart con Python y Pandas

**Evidencia:** GA1-220501094-01-AA1-EV04 – Fundamentos en Pandas Explorando Instacart: descubre patrones de compra  
**Aprendiz:** escribe aquí tu nombre  
**Programa / ficha:** completa estos datos antes de entregar.

Este Notebook explora los cinco archivos entregados, revisa su estructura, aplica indexación y filtros, y calcula indicadores descriptivos.

## Momento 1. Comprensión del problema y reconocimiento de los datos

Instacart registra información sobre pedidos de supermercado, los usuarios que los realizan, los productos disponibles y los artículos que se agregan a cada pedido. El equipo de análisis necesita conocer qué contienen los archivos y cómo consultarlos para identificar patrones descriptivos de compra.

La pregunta orientadora es: **¿Cómo utilizar Python y Pandas para explorar los datos de Instacart, identificar las características de los pedidos y productos, y obtener información relevante sobre los hábitos de compra de sus clientes?**

Los archivos representan entidades diferentes: `instacart_orders.csv` contiene atributos de pedidos; `products.csv` describe el catálogo; `aisles.csv` y `departments.csv` contienen nombres de categorías; y `order_products.csv` relaciona artículos con pedidos e incluye información de posición en el carrito y recompra. Identificadores como `order_id`, `user_id`, `product_id`, `aisle_id` y `department_id` pueden relacionar conceptualmente esas tablas cuando están disponibles.

### Preguntas iniciales
1. ¿En qué hora del día se registra la mayor cantidad de pedidos?
2. ¿Cuántos registros de artículos están marcados como recompra y qué proporción representan?
3. ¿Cuál es el promedio de días transcurridos entre pedidos, usando los valores disponibles?

Explorar antes de analizar permite comprobar la carga y las dimensiones, entender qué representa cada fila y columna, reconocer tipos de datos y localizar valores ausentes. Así se reducen errores de interpretación, por ejemplo confundir filas de pedidos con usuarios únicos.

## Momento 2. Importación y exploración inicial

Los cinco CSV se encuentran junto al Notebook. Se usa obligatoriamente `sep=";"`, pues ese es el separador indicado para estos archivos. Si los guardaste en otra carpeta, modifica `DATA_DIR`.

**En Google Colab:** sube los 5 archivos CSV a la sesión (panel de archivos o con el widget que aparece al ejecutar la celda de carga) o colócalos en `/content`. La celda de carga los busca automáticamente en `.`, `/content` y `/content/drive/MyDrive`.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Configuracion de estilo para las visualizaciones
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

ARCHIVOS = [
    "instacart_orders.csv", "products.csv", "aisles.csv",
    "departments.csv", "order_products.csv",
]

# Busqueda rapida en rutas comunes (sin recorrer todo Drive).
def _buscar_carpeta():
    raices = [Path("../data"), Path("data"), Path("."), Path("/content"), Path("/content/data")]
    mydrive = Path("/content/drive/MyDrive")
    if mydrive.exists():
        raices.append(mydrive)
        raices.append(mydrive / "data")
    for raiz in raices:
        if not raiz.exists():
            continue
        if all((raiz / a).exists() for a in ARCHIVOS):
            return raiz
        try:
            for sub in raiz.iterdir():
                if sub.is_dir() and all((sub / a).exists() for a in ARCHIVOS):
                    return sub
        except Exception:
            pass
    return None

if not all((DATA_DIR / a).exists() for a in ARCHIVOS):
    encontrada = _buscar_carpeta()
    if encontrada is not None:
        DATA_DIR = encontrada
    else:
        try:
            from google.colab import files
            print("Sube los 5 archivos CSV (instacart_orders, products, aisles, departments, order_products):")
            files.upload()
            DATA_DIR = Path(".")
        except Exception:
            raise FileNotFoundError(
                "No se encontraron los CSV. Colocalos en data/ o sube los 5 archivos a Colab."
            )

orders = pd.read_csv(DATA_DIR / "instacart_orders.csv", sep=";", encoding="utf-8-sig")
products = pd.read_csv(DATA_DIR / "products.csv", sep=";", encoding="utf-8-sig")
aisles = pd.read_csv(DATA_DIR / "aisles.csv", sep=";", encoding="utf-8-sig")
departments = pd.read_csv(DATA_DIR / "departments.csv", sep=";", encoding="utf-8-sig")
order_products = pd.read_csv(DATA_DIR / "order_products.csv", sep=";", encoding="utf-8-sig")

print("Los cinco archivos se cargaron correctamente.")
print(f"DATA_DIR = {DATA_DIR}")

dataframes = {
    "orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}


Se muestran los primeros y últimos cinco registros de cada conjunto, además de sus dimensiones en el orden `(filas, columnas)`. En tablas grandes se limita la vista a esas muestras para que la salida sea manejable.

In [ ]:
for nombre, df in dataframes.items():
    print(f"\n{'=' * 72}\n{nombre}: primeros cinco registros")
    display(df.head())
    print(f"{nombre}: últimos cinco registros")
    display(df.tail())
    print(f"{nombre}: shape = {df.shape} (filas, columnas)")

In [ ]:
nombre_mas_filas = max(dataframes, key=lambda nombre: dataframes[nombre].shape[0])
maximo_filas = dataframes[nombre_mas_filas].shape[0]
minimo_columnas = min(df.shape[1] for df in dataframes.values())
nombres_menos_columnas = [
    nombre for nombre, df in dataframes.items() if df.shape[1] == minimo_columnas
]

print(f"Mayor cantidad de registros: {nombre_mas_filas} ({maximo_filas:,} filas).")
print(
    f"Menor cantidad de columnas: {minimo_columnas}; "
    f"DataFrame(s): {', '.join(nombres_menos_columnas)}."
)

### Análisis de las dimensiones

El bloque anterior determina estos resultados desde los archivos cargados; según las dimensiones de referencia de la guía, `order_products` es el conjunto con más filas y `aisles` y `departments` comparten la menor cantidad de columnas (dos). `orders` describe pedidos, mientras que `products` describe elementos del catálogo: sus filas no representan el mismo tipo de entidad. Conocer las dimensiones ayuda a comprender el volumen, comparar conjuntos y planear el procesamiento; no permite por sí solo deducir cuántos pedidos o usuarios únicos existen.

## Momento 3. Estructura y calidad inicial de los datos

Se inspeccionarán nombres de columnas, tipos, dimensiones y valores no nulos. En esta fase se identifican posibles dificultades, pero no se eliminan ni se modifican registros.

In [ ]:
for nombre, df in dataframes.items():
    print(f"\n{'=' * 72}\nDataFrame: {nombre}")
    print("Columnas:", df.columns.tolist())
    print("\nTipos de datos:\n", df.dtypes)
    print("\nDimensiones:", df.shape)
    print("\nResumen de columnas y valores no nulos:")
    df.info()

In [ ]:
dimensiones_referencia = {
    "orders": (478_967, 6),
    "products": (49_694, 4),
    "aisles": (134, 2),
    "departments": (21, 2),
    "order_products": (4_545_007, 4),
}

filas_diagnostico = []
for nombre, df in dataframes.items():
    conteo_nulos = df.isna().sum()
    columnas_con_nulos = conteo_nulos[conteo_nulos > 0]
    if columnas_con_nulos.empty:
        observacion_nulos = "No se detectan valores ausentes." 
    else:
        detalle = ", ".join(
            f"{col}: {cantidad:,}" for col, cantidad in columnas_con_nulos.items()
        )
        observacion_nulos = f"Valores ausentes — {detalle}. "

    coincide = df.shape == dimensiones_referencia[nombre]
    observacion_dimensiones = (
        "Dimensiones coinciden con la guía."
        if coincide
        else f"Dimensiones distintas a la guía: {dimensiones_referencia[nombre]}."
    )
    filas_diagnostico.append(
        {
            "Nombre del archivo": f"{nombre}.csv",
            "Cantidad de filas": df.shape[0],
            "Cantidad de columnas": df.shape[1],
            "Observaciones": observacion_nulos + observacion_dimensiones,
        }
    )

diagnostico = pd.DataFrame(filas_diagnostico)
display(diagnostico)

In [ ]:
for nombre, df in dataframes.items():
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0]
    print(f"\nValores ausentes en {nombre}:")
    if nulos.empty:
        print("No se encontraron valores ausentes.")
    else:
        display(nulos.rename("cantidad de valores ausentes").to_frame())

### Interpretación de tipos y calidad

- `int64` representa números enteros y `float64` valores numéricos de punto flotante. Una columna con datos ausentes puede almacenarse como decimal aunque los valores observados sean enteros.
- `object` suele representar texto, aunque puede contener valores de tipos mixtos.
- `shape` presenta únicamente filas y columnas. `info()` agrega nombres, tipos y conteos de valores no nulos, útiles para localizar ausencias.
- La tabla y los conteos anteriores identifican las columnas con valores ausentes a partir de los datos reales. Los ausentes pueden alterar promedios o provocar problemas en filtros y conversiones. La media de Pandas omite `NaN` por defecto; cualquier imputación o eliminación debe justificarse según el significado del campo.
- Las relaciones conceptuales se establecen mediante identificadores compartidos: por ejemplo `order_id` relaciona pedidos con artículos, `product_id` relaciona artículos con el catálogo y `aisle_id` / `department_id` conectan productos con sus clasificaciones.

## Momento 4. Indexación y filtrado lógico

Seleccionar columnas conserva las variables solicitadas; filtrar filas conserva los registros que cumplen una condición. `loc[filas, columnas]` permite expresar ambas selecciones por etiquetas o con una máscara booleana. Para combinar condiciones se usan `&` (AND) y `|` (OR), con cada condición entre paréntesis. AND requiere que todas se cumplan, por lo que el conjunto resultante normalmente tiene igual o menos filas que cada filtro individual.

In [ ]:
# 1. Seleccionar las columnas product_id y product_name.
productos_id_nombre = products.loc[:, ["product_id", "product_name"]]
print(f"1. Selección de columnas: {len(productos_id_nombre):,} filas")
display(productos_id_nombre.head())

# 2. Productos del departamento 1.
productos_depto_1 = products.loc[products["department_id"] == 1]
print(f"2. department_id == 1: {len(productos_depto_1):,} filas")
display(productos_depto_1.head())

# 3. Pedidos realizados a las 10 horas.
pedidos_hora_10 = orders.loc[orders["order_hour_of_day"] == 10]
print(f"3. order_hour_of_day == 10: {len(pedidos_hora_10):,} filas")
display(pedidos_hora_10.head())

# 4. Pedidos cuyo día de la semana es 0.
pedidos_dia_0 = orders.loc[orders["order_dow"] == 0]
print(f"4. order_dow == 0: {len(pedidos_dia_0):,} filas")
display(pedidos_dia_0.head())

# 5. Pedidos con número superior a 10.
pedidos_numero_mayor_10 = orders.loc[orders["order_number"] > 10]
print(f"5. order_number > 10: {len(pedidos_numero_mayor_10):,} filas")
display(pedidos_numero_mayor_10.head())

# 6. Más de 15 días desde el pedido anterior.
pedidos_intervalo_mayor_15 = orders.loc[
    orders["days_since_prior_order"] > 15
]
print(
    "6. days_since_prior_order > 15: "
    f"{len(pedidos_intervalo_mayor_15):,} filas"
)
display(pedidos_intervalo_mayor_15.head())

# 7. Registros de artículos marcados como recompra.
articulos_recomprados = order_products.loc[order_products["reordered"] == 1]
print(f"7. reordered == 1: {len(articulos_recomprados):,} filas")
display(articulos_recomprados.head())

# 8. Artículos incorporados en la primera posición del carrito.
articulos_primera_posicion = order_products.loc[
    order_products["add_to_cart_order"] == 1
]
print(
    "8. add_to_cart_order == 1: "
    f"{len(articulos_primera_posicion):,} filas"
)
display(articulos_primera_posicion.head())

In [ ]:
# Pedidos a las 10 horas Y con order_number superior a 10.
mascara_pedidos = (
    (orders["order_hour_of_day"] == 10)
    & (orders["order_number"] > 10)
)
pedidos_hora_10_y_numero_mayor_10 = orders.loc[mascara_pedidos]
print(
    "Pedidos a las 10 h Y order_number > 10: "
    f"{len(pedidos_hora_10_y_numero_mayor_10):,} filas"
)
display(pedidos_hora_10_y_numero_mayor_10.head())

# Productos del departamento 1 Y con aisle_id superior a 50.
mascara_productos = (
    (products["department_id"] == 1)
    & (products["aisle_id"] > 50)
)
productos_depto_1_pasillo_mayor_50 = products.loc[mascara_productos]
print(
    "Productos department_id == 1 Y aisle_id > 50: "
    f"{len(productos_depto_1_pasillo_mayor_50):,} filas"
)
display(productos_depto_1_pasillo_mayor_50.head())

Cada consulta presenta el total de filas que cumple el criterio y una muestra de hasta cinco resultados. Esto responde a la guía sin saturar la salida, especialmente en `order_products`, que contiene millones de registros. Para OR, `|` incluiría filas que cumplan al menos una condición; es importante usar operadores de Pandas y paréntesis para evitar errores de precedencia.

## Momento 5. Obtención e interpretación de indicadores básicos

Se calculan métricas de filas, promedios, recompra y estadísticas descriptivas. Se distingue explícitamente el número de registros de pedidos, los identificadores únicos de pedido y usuario y las filas de artículos.

In [ ]:
filas_orders = len(orders)
filas_products = len(products)
filas_order_products = len(order_products)
promedio_order_number = orders["order_number"].mean()
promedio_dias_entre_pedidos = orders["days_since_prior_order"].mean()
cantidad_recompras = int(order_products["reordered"].eq(1).sum())
suma_reordered = order_products["reordered"].sum()
proporcion_recompras = (
    cantidad_recompras / filas_order_products if filas_order_products else float("nan")
)

estadisticas_hora = orders["order_hour_of_day"].describe()
estadisticas_carrito = order_products["add_to_cart_order"].describe()

print("Estadísticas de order_hour_of_day:")
display(estadisticas_hora.to_frame(name="valor"))
print("Estadísticas de add_to_cart_order:")
display(estadisticas_carrito.to_frame(name="valor"))

In [ ]:
indicadores = [
    {
        "Pregunta": "¿Cuántos registros de pedidos contiene el archivo?",
        "Valor obtenido": f"{filas_orders:,} filas",
        "Interpretación": "Número de filas en orders; no significa necesariamente usuarios únicos.",
    },
    {
        "Pregunta": "¿Cuántos registros contiene el catálogo de productos?",
        "Valor obtenido": f"{filas_products:,} filas",
        "Interpretación": "Cantidad de filas del catálogo products.",
    },
    {
        "Pregunta": "¿Cuál es el promedio de order_number?",
        "Valor obtenido": f"{promedio_order_number:.2f}",
        "Interpretación": "Media del número de pedido en las filas disponibles de orders.",
    },
    {
        "Pregunta": "¿Cuál es el promedio de días desde el pedido anterior?",
        "Valor obtenido": f"{promedio_dias_entre_pedidos:.2f}",
        "Interpretación": "La media de Pandas omite los valores ausentes de esta columna.",
    },
    {
        "Pregunta": "¿Cuántos registros tienen reordered igual a 1?",
        "Valor obtenido": f"{cantidad_recompras:,}",
        "Interpretación": "Cuenta filas de artículos marcadas como recompra, no usuarios.",
    },
    {
        "Pregunta": "¿Cuál es la suma total de reordered?",
        "Valor obtenido": f"{suma_reordered:,}",
        "Interpretación": "Con valores binarios 0/1, la suma coincide con el conteo de unos.",
    },
    {
        "Pregunta": "¿Qué describe order_hour_of_day?",
        "Valor obtenido": str(estadisticas_hora.round(2).to_dict()),
        "Interpretación": "count, media, desviación estándar, mínimo, cuartiles y máximo.",
    },
    {
        "Pregunta": "¿Qué describe add_to_cart_order?",
        "Valor obtenido": str(estadisticas_carrito.round(2).to_dict()),
        "Interpretación": "Resumen estadístico de las posiciones de artículos en el carrito.",
    },
]

if "order_id" in orders.columns:
    pedidos_unicos = orders["order_id"].nunique()
    indicadores.append(
        {
            "Pregunta": "¿Cuántos pedidos únicos hay según order_id?",
            "Valor obtenido": f"{pedidos_unicos:,}",
            "Interpretación": "Conteo de identificadores de pedido distintos.",
        }
    )
if "user_id" in orders.columns:
    usuarios_unicos = orders["user_id"].nunique()
    indicadores.append(
        {
            "Pregunta": "¿Cuántos usuarios únicos hay según user_id?",
            "Valor obtenido": f"{usuarios_unicos:,}",
            "Interpretación": "Conteo de identificadores de usuario distintos.",
        }
    )

indicadores.append(
    {
        "Pregunta": "¿Qué proporción de registros de artículos es recompra?",
        "Valor obtenido": f"{proporcion_recompras:.2%}",
        "Interpretación": f"{cantidad_recompras:,} de {filas_order_products:,} filas de order_products.",
    }
)

tabla_indicadores = pd.DataFrame(indicadores)
display(tabla_indicadores)

### Cómo interpretar los conteos

`len(orders)` cuenta filas de pedidos. `nunique()` sobre `order_id` cuenta pedidos identificados de forma distinta, y sobre `user_id` cuenta usuarios distintos. `len(order_products)` cuenta filas de artículos relacionados con pedidos; una persona o pedido puede aparecer en varias filas. Por eso estos conteos no son intercambiables.

## Momento 6. Visualizacion de datos

La visualizacion permite identificar patrones, distribuciones y relaciones que no son evidentes en tablas numericas. Se usan matplotlib y seaborn.


In [ ]:
# 1. Histograma: pedidos por hora del dia
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(orders['order_hour_of_day'], bins=24, color='steelblue', ax=ax)
ax.set_title('Distribucion de pedidos por hora del dia')
ax.set_xlabel('Hora del dia')
ax.set_ylabel('Cantidad de pedidos')
plt.tight_layout()
plt.show()

# 2. Distribucion de dias entre pedidos
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(orders['days_since_prior_order'].dropna(), bins=30, kde=True, color='coral', ax=ax)
ax.set_title('Distribucion de dias transcurridos entre pedidos')
ax.set_xlabel('Dias desde el pedido anterior')
ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

# 3. Proporcion de articulos recomprados
fig, ax = plt.subplots(figsize=(6, 6))
reordered_counts = order_products['reordered'].value_counts().sort_index()
labels = ['No recomprado (0)', 'Recomprado (1)']
colors = ['#66b3ff', '#99ff99']
ax.pie(reordered_counts, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
ax.set_title('Proporcion de articulos recomprados')
plt.tight_layout()
plt.show()

# 4. Top 10 productos mas frecuentes
top_productos = order_products['product_id'].value_counts().head(10).reset_index()
top_productos.columns = ['product_id', 'cantidad']
top_productos = top_productos.merge(products[['product_id', 'product_name']], on='product_id', how='left')
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top_productos, y='product_name', x='cantidad', palette='viridis', ax=ax)
ax.set_title('Top 10 productos mas frecuentes en pedidos')
ax.set_xlabel('Cantidad de apariciones')
ax.set_ylabel('Producto')
plt.tight_layout()
plt.show()

# 5. Top 10 departamentos con mas productos
depto_counts = products['department_id'].value_counts().head(10).reset_index()
depto_counts.columns = ['department_id', 'cantidad_productos']
depto_counts = depto_counts.merge(departments, on='department_id', how='left')
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=depto_counts, y='department', x='cantidad_productos', palette='magma', ax=ax)
ax.set_title('Top 10 departamentos con mas productos en el catalogo')
ax.set_xlabel('Cantidad de productos')
ax.set_ylabel('Departamento')
plt.tight_layout()
plt.show()

# 6. Boxplot: hora vs numero de pedido
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=orders, x='order_hour_of_day', y='order_number', palette='coolwarm', ax=ax)
ax.set_title('Distribucion del numero de pedido por hora del dia')
ax.set_xlabel('Hora del dia')
ax.set_ylabel('Numero de pedido')
plt.tight_layout()
plt.show()

# 7. Heatmap de correlacion
fig, ax = plt.subplots(figsize=(8, 6))
corr = orders.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Matriz de correlacion - orders')
plt.tight_layout()
plt.show()


## Momento 7. Analisis avanzado con Pandas

Se aplican operaciones mas avanzadas: groupby, merge y pivot_table.


In [ ]:
# 1. Merge de tablas
order_products_full = (
    order_products
    .merge(products, on='product_id', how='left')
    .merge(aisles, on='aisle_id', how='left')
    .merge(departments, on='department_id', how='left')
)
print(f'Tabla combinada: {order_products_full.shape[0]:,} filas')
display(order_products_full.head())

# 2. Groupby: articulos por departamento
articulos_por_depto = (
    order_products_full
    .groupby('department')['product_id']
    .count()
    .sort_values(ascending=False)
    .reset_index(name='cantidad_articulos')
)
display(articulos_por_depto)

# 3. Tasa de recompra por departamento
recompra_por_depto = (
    order_products_full
    .groupby('department')['reordered']
    .mean()
    .sort_values(ascending=False)
    .reset_index(name='tasa_recompra')
)
recompra_por_depto['tasa_recompra'] = recompra_por_depto['tasa_recompra'].round(4)
display(recompra_por_depto)

# 4. Pivot table: pedidos por dia y hora
pivot_hora_dia = orders.pivot_table(
    index='order_dow', columns='order_hour_of_day',
    values='order_id', aggfunc='count', fill_value=0,
)
display(pivot_hora_dia)
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot_hora_dia, cmap='YlOrRd', annot=False, ax=ax)
ax.set_title('Mapa de calor: pedidos por dia de la semana y hora')
plt.tight_layout()
plt.show()

# 5. Top 10 productos mas recomprados
top_recomprados = (
    order_products_full[order_products_full['reordered'] == 1]
    .groupby('product_name')['reordered']
    .count()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='veces_recomprado')
)
display(top_recomprados)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top_recomprados, y='product_name', x='veces_recomprado', palette='rocket', ax=ax)
ax.set_title('Top 10 productos mas recomprados')
plt.tight_layout()
plt.show()

# 6. Estadisticas por departamento
stats_depto = (
    order_products_full
    .groupby('department')
    .agg(total_articulos=('product_id', 'count'), productos_unicos=('product_id', 'nunique'), tasa_recompra=('reordered', 'mean'))
    .round(4)
    .sort_values('total_articulos', ascending=False)
)
display(stats_depto)


## Momento 8. Resolucion del problema y presentacion del producto

La sintesis final se construye con los resultados calculados por el Notebook.


In [ ]:
modas_hora = orders["order_hour_of_day"].mode().tolist()
horas_moda_texto = ", ".join(str(int(hora)) for hora in modas_hora)
modas_dia = orders["order_dow"].mode().tolist()
dias_moda_texto = ", ".join(str(int(dia)) for dia in modas_dia)
nulos_totales = sum(int(df.isna().sum().sum()) for df in dataframes.values())

sintesis = f"""
### Respuestas a las preguntas iniciales

1. **¿En qué hora del día se registra la mayor cantidad de pedidos?** La(s) hora(s) con mayor frecuencia en `order_hour_of_day` es/son **{horas_moda_texto}**. La moda puede tener más de un valor si hay empate.
2. **¿Cuántos artículos están marcados como recompra?** Hay **{cantidad_recompras:,}** registros con `reordered == 1`, equivalentes al **{proporcion_recompras:.2%}** de las **{filas_order_products:,}** filas de `order_products`.
3. **¿Cuál es el promedio de días entre pedidos?** El promedio de `days_since_prior_order` es **{promedio_dias_entre_pedidos:.2f} días**, calculado con los valores no ausentes.

### Hallazgos y respuesta orientadora

Con Python y Pandas se cargaron los cinco archivos con separador punto y coma, se revisaron sus muestras, dimensiones, columnas, tipos y valores ausentes, y se ejecutaron consultas por condiciones. `orders` contiene **{filas_orders:,} filas** y `products` **{filas_products:,} filas**; `order_products` registra **{filas_order_products:,} filas de artículos**, por lo que es importante no confundir artículos con pedidos o clientes. La hora de mayor frecuencia observada es **{horas_moda_texto}**. La exploración muestra que los datos se pueden consultar para describir características y hábitos registrados, aunque las conclusiones se limitan a los archivos disponibles.

La indexación con `loc[]` seleccionó columnas o filas sin necesidad de imprimir el conjunto completo. Las máscaras booleanas localizaron pedidos, productos y artículos que cumplen criterios simples o simultáneos. AND (`&`) permitió restringir el resultado a filas que cumplen todas las condiciones.

### Limitaciones

- Se identificaron **{nulos_totales:,} valores ausentes en total** entre los cinco DataFrames; el diagnóstico muestra las columnas afectadas. El promedio de días excluye los ausentes por el comportamiento predeterminado de Pandas.
- Los resultados son descriptivos del contenido suministrado y no demuestran causas ni necesariamente representan otros periodos o la totalidad de clientes.
- Una fila de cada tabla representa unidades distintas: pedido, producto del catálogo o artículo dentro de un pedido. Los conteos deben interpretarse según esa unidad.
- No se corrigieron ni imputaron datos. Para un análisis posterior, habría que justificar cualquier tratamiento de ausentes y validar las relaciones entre identificadores.

### Tres conclusiones

1. **La exploración aporta contexto antes de interpretar métricas:** el tamaño más grande y el menor número de columnas se muestran en Momento 2, mientras que el diagnóstico de Momento 3 permite verificar dimensiones y ausencias observadas.
2. **Los filtros permiten responder consultas concretas:** las cantidades impresas en Momento 4 muestran cuántas filas cumplen cada condición, y las combinaciones AND delimitan aún más el conjunto de resultados.
3. **Las métricas de recompra y frecuencia son descriptivas y dependen de la unidad de análisis:** la cantidad de filas con `reordered == 1` describe registros de artículos; la media de `days_since_prior_order` resume los valores disponibles, no el comportamiento de cada usuario individual.
"""

display(Markdown(sintesis))

## Verificación para la entrega

1. Ejecuta **Run All** y confirma que no haya errores.
2. Verifica que `DATA_DIR` apunte a la carpeta donde están los CSV y que las dimensiones reportadas sean coherentes con la guía.
3. Completa tu nombre, programa y ficha en la portada.
4. Revisa que las salidas de indicadores y las conclusiones correspondan a los resultados que produjo tu ejecución.
5. Guarda el archivo `.ipynb` con las salidas si así lo solicita tu instructor.